In [1]:
# Script to convert all the h5 files to save out just the lat, lon, outer surface Br and date
# 
# These are the DeRosa ones that are part of the sswidl pfss package
# https://hesperia.gsfc.nasa.gov/ssw/packages/pfss/
# https://www.lmsal.com/~derosa/pfsspack/
# 
# Files urls are like:
# https://www.lmsal.com/solarsoft/archive/ssw/pfss_links_v2/Bfield_20030607_120400.h5
# 
# 6 Oct 2026 IGH

In [6]:
import h5py
import astropy
import numpy as np
import calendar
import subprocess
from pathlib import Path

outdir = Path("br_data")

for fname in sorted(Path("pfss_org").rglob("Bfield*.h5")):
    with h5py.File(fname) as f:
        d = f["ssw_pfss_extrapolation"][0]
    lat, lon, L0 = d["LAT"], d["LON"], d["L0"]
    date = astropy.time.Time(d["MODEL_DATE"].decode())

    dlon = (lon - L0 + 180) % 360 - 180
    order = np.argsort(dlon)
    br_out = d["BR"][-1][:, order]

    # Save just what the later plots need (lat/lon grid, Br at the outer surface,
    # and the date) as a compressed .npz file, organised into a subdirectory per
    # year and month (e.g. pfss_data/2026/03). This is a plain numpy format: much
    # smaller/faster to read back than hdf5 for arrays this size, and
    # safer/more portable than pickle (no arbitrary code execution on load,
    # no python-version dependence).
    yeardir = outdir / date.isot[:4] / date.isot[5:7]
    yeardir.mkdir(parents=True, exist_ok=True)
    outname = yeardir / f"brout_{date.isot[:19].replace('-', '').replace(':', '')}.npz"
    np.savez_compressed(outname, lat=lat, lon=dlon[order], br_out=br_out, date=date.isot)
